Welcome to Pipelines!

The HuggingFace transformers library provides APIs at two different levels.

The High Level API for using open-source models for typical inference tasks is called "pipelines". It's incredibly easy to use.

You create a pipeline using something like:

my_pipeline = pipeline("the_task_I_want_to_do")

Followed by

result = my_pipeline(my_input)

And that's it!

See end of this colab for a list of all pipelines.


## Before we start: 2 important pro-tips for using Colab:

**Pro-tip 1:**

Data Science code often gives warnings and messages. They can mostly be safely ignored! Glance over them, and if something goes wrong later, perhaps they can give you a clue.

**Pro-tip 2:**

In the middle of running a Colab, you might get an error like this:

> Runtime error: CUDA is required but not available for bitsandbytes. Please consider installing [...]

This is a super-misleading error message! Please don't try changing versions of packages...

This actually happens because Google has switched out your Colab runtime, perhaps because Google Colab was too busy. The solution is:

1. Kernel menu >> Disconnect and delete runtime
2. Reload the colab from fresh and Edit menu >> Clear All Outputs
3. Connect to a new T4 using the button at the top right
4. Select "View resources" from the menu on the top right to confirm you have a GPU
5. Rerun the cells in the colab, from the top down, starting with the pip installs


## Pipelines

Used for simple out-of-the-box inference tasks


*   Sentiment Analysis
*   Classifier
*   NER
*   Q&A
*   Summarising
*   Translation



In [2]:
pip uninstall -q -y torch torchvision torchaudio

In [5]:
pip install -q torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121

In [6]:
#pip installs always come first
#rerun when restarting kernal
#upgrade for if it is already installed and -q for quiet
!pip install -q --upgrade datasets==3.6.0 transformers==4.57.6 #pin the package to a version


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 1.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 491.5/491.5 kB 13.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 68.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 27.0 MB/s eta 0:00:00


In [5]:
#check gpu
gpu_info = !nvidia-smi
gpu_info = '\n'.join(gpu_info)

if gpu_info.find('failed') >=0:
  print("Not connected to a GPU")
else:
  print(gpu_info)
  if gpu_info.find('Tesla T4') >=0:
    print("Connected to T4 GPU")
  else:
    print("NOT CONNECTED TO T4")

Mon Apr 27 11:03:38 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   36C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [3]:
#imports
import torch
from google.colab import userdata
from huggingface_hub import login
from transformers import pipeline
from diffusers import DiffusionPipeline
from datasets import load_dataset
import soundfile as sf
from IPython.display import Audio

Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.
Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.


In [5]:
hf_token = userdata.get("HF_TOKEN")

if hf_token and hf_token.startswith("hf"):
  print("Valid HF Token")
else:
  print("Invlaid HF Token")

login(hf_token, add_to_git_credential=True)

Valid HF Token


## Using Pipelines from Hugging Face

A simple way to run inference for common tasks, without worrying about all the plumbing, picking reasonable defaults.


### How it works:

STEP 1: Create a pipeline - a function you can then call

```python
my_pipeline = pipeline(task, model=xx, device=xx)
```

If you don't specify a model, then Hugging Face picks one for you that's the default for the task. Specify "cuda" for the device to use an NVIDIA GPU like the one on the T4. Specify "mps" on a Mac.


STEP 2: Then call it as many times as you want:

```python
my_pipeline(input1)
my_pipeline(input2)
```

In [6]:
#sentiment analysis
sentiment_analyser = pipeline("sentiment-analysis", device = "cuda")
result = sentiment_analyser("I love this course! I am super excited to be on the way to LLM Mastery")
print(result)
#

No model was supplied, defaulted to distilbert/distilbert-base-uncased-finetuned-sst-2-english and revision 714eb0f.
Using a pipeline without specifying a model name and revision in production is not recommended.


config.json:   0%|          | 0.00/629 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

[{'label': 'POSITIVE', 'score': 0.9998676776885986}]


In [7]:
result = sentiment_analyser("I dont like my job. The work is boring and my boss is rude")
print(result)

[{'label': 'NEGATIVE', 'score': 0.999624490737915}]


In [8]:
better_sent_analysis = pipeline("sentiment-analysis", model = "cardiffnlp/twitter-roberta-base-sentiment-latest", device="cuda")
result = better_sent_analysis("I should be more xcited to become an LLM Master but it is hard work. Im sure it will be rewarding in the end")
print(result)

config.json:   0%|          | 0.00/929 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/501M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

RobertaForSequenceClassification LOAD REPORT from: cardiffnlp/twitter-roberta-base-sentiment-latest
Key                             | Status     |  | 
--------------------------------+------------+--+-
roberta.pooler.dense.weight     | UNEXPECTED |  | 
roberta.embeddings.position_ids | UNEXPECTED |  | 
roberta.pooler.dense.bias       | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


model.safetensors:   0%|          | 0.00/501M [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

[{'label': 'positive', 'score': 0.8819913268089294}]


In [10]:
#NER - splits by token
ner = pipeline("ner", device="cuda")
result=ner("AI Engineers learning about pipelines, HuggingFace, and LLms from Ed Donner")
for entity in result:
  print(entity)

No model was supplied, defaulted to dbmdz/bert-large-cased-finetuned-conll03-english and revision 4c53496.
Using a pipeline without specifying a model name and revision in production is not recommended.


Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

BertForTokenClassification LOAD REPORT from: dbmdz/bert-large-cased-finetuned-conll03-english
Key                      | Status     |  | 
-------------------------+------------+--+-
bert.pooler.dense.weight | UNEXPECTED |  | 
bert.pooler.dense.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


{'entity': 'I-ORG', 'score': np.float32(0.9995683), 'index': 1, 'word': 'AI', 'start': 0, 'end': 2}
{'entity': 'I-ORG', 'score': np.float32(0.9975604), 'index': 2, 'word': 'Engineers', 'start': 3, 'end': 12}
{'entity': 'I-ORG', 'score': np.float32(0.7050881), 'index': 9, 'word': '##gging', 'start': 41, 'end': 46}
{'entity': 'I-ORG', 'score': np.float32(0.8301573), 'index': 10, 'word': '##F', 'start': 46, 'end': 47}
{'entity': 'I-ORG', 'score': np.float32(0.78285104), 'index': 11, 'word': '##ace', 'start': 47, 'end': 50}
{'entity': 'I-ORG', 'score': np.float32(0.5042004), 'index': 14, 'word': 'LL', 'start': 56, 'end': 58}
{'entity': 'I-PER', 'score': np.float32(0.9988257), 'index': 17, 'word': 'Ed', 'start': 66, 'end': 68}
{'entity': 'I-PER', 'score': np.float32(0.9991341), 'index': 18, 'word': 'Don', 'start': 69, 'end': 72}
{'entity': 'I-PER', 'score': np.float32(0.9811257), 'index': 19, 'word': '##ner', 'start': 72, 'end': 75}


In [14]:
# baby step to rag - in practice use ner to get relevant docs from database to use as context for the question
question = "What are Hugging Face pipelines?"
context = "Pipelines are high level API for inference of LLMs with common tasks"
question_answerer = pipeline("question-answering", device="cuda")
result=question_answerer(question=question, context=context)
print(result)

No model was supplied, defaulted to distilbert/distilbert-base-cased-distilled-squad and revision 564e9b5.
Using a pipeline without specifying a model name and revision in production is not recommended.


Loading weights:   0%|          | 0/102 [00:00<?, ?it/s]

{'score': 0.4247668385505676, 'start': 14, 'end': 68, 'answer': 'high level API for inference of LLMs with common tasks'}


In [2]:
from transformers import pipeline

summariser = pipeline(
    "summarization",
    model="facebook/bart-large-cnn",
    device=0
)

text = """
The Hugging Face transformers library is an incredibly versatile and powerful tool for natural language processing (NLP).
It allows users to perform a wide range of tasks such as text classification, named entity recognition, and question answering, among others.
It's an extremely popular library that's widely used by the open-source data science community.
It lowers the barrier to entry into the field by providing Data Scientists with a productive, convenient way to work with transformer models.
"""


print(summariser(text, max_length=100, min_length=30, do_sample=False)[0]["summary_text"])

model.safetensors:   0%|          | 0.00/1.63G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/363 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Device set to use cuda:0


The Hugging Face transformers library is an incredibly versatile and powerful tool for natural language processing. It allows users to perform a wide range of tasks such as text classification, named entity recognition and question answering.


In [1]:
#translation
from transformers import pipeline

translator = pipeline("translation_en_to_fr", device="cuda")
result = translator("The data scientists were truly amazed by the power of AI")
print(result[0]['translation_text'])

No model was supplied, defaulted to google-t5/t5-base and revision a9723ea (https://huggingface.co/google-t5/t5-base).
Using a pipeline without specifying a model name and revision in production is not recommended.


config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/892M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Device set to use cuda


Les scientifiques des données ont été vraiment étonnés par la puissance de l’IA.


In [ ]:
#small specialised model
translator = pipeline("translation_en_to_es", model="Helsinki-NLP/opus-mt-en-es", device="cuda")
result = translator("The Data Scientists were truly amazed by the power and simplicity of the HuggingFace pipeline API.")
print(result[0]['translation_text'])

In [2]:
classifier = pipeline("zero-shot-classification", device="cuda")
result = classifier("Hugging face Transformers library is a great starting point for findng the right model", candidate_labels = ["tech", "sport", "food"])
print(result)

No model was supplied, defaulted to facebook/bart-large-mnli and revision d7645e1 (https://huggingface.co/facebook/bart-large-mnli).
Using a pipeline without specifying a model name and revision in production is not recommended.


config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/1.63G [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Device set to use cuda


{'sequence': 'Hugging face Transformers library is a great starting point for findng the right model', 'labels': ['tech', 'sport', 'food'], 'scores': [0.8881680369377136, 0.07776249945163727, 0.034069448709487915]}
